<a href="https://colab.research.google.com/github/PC-Tam/GTSRB-DeepLearning-Classification/blob/main/notebooks/section1_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install torch torchvision
%pip install pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\pct2k\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\pct2k\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
# ============================================================
# Tải và giải nén dataset GTSRB từ Kaggle (chỉ chạy 1 lần)
# ============================================================
# Yêu cầu: file kaggle.json đặt tại C:\Users\<username>\.kaggle\kaggle.json
# Lấy API key tại: https://www.kaggle.com/settings → Create New Token

import os
import zipfile
from pathlib import Path

# Tự động xác định BASE_DIR
if os.path.exists(os.path.join(os.getcwd(), "notebooks")):
    BASE_DIR = os.getcwd()
elif os.path.basename(os.getcwd()) == "notebooks":
    BASE_DIR = str(Path(os.getcwd()).parent)
else:
    BASE_DIR = os.getcwd()

EXTRACTED_DIR = os.path.join(BASE_DIR, "data", "extracted")
TRAIN_DIR = os.path.join(EXTRACTED_DIR, "Train")

# Kiểm tra nếu đã có đủ 43 class thì bỏ qua
need_download = True
if os.path.exists(TRAIN_DIR):
    existing_classes = [d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))]
    if len(existing_classes) >= 43:
        total_imgs = sum(len(os.listdir(os.path.join(TRAIN_DIR, d))) for d in existing_classes)
        print(f"Dataset đã tồn tại đầy đủ: {len(existing_classes)} class, {total_imgs} ảnh.")
        print("Bỏ qua bước tải.")
        need_download = False
    else:
        print(f"Chỉ có {len(existing_classes)}/43 class. Đang tải lại...")

if need_download:
    # Cài kaggle nếu chưa có
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle"])

    from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate()

    DOWNLOAD_DIR = os.path.join(BASE_DIR, "data", "raw")
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)

    DATASET = "meowmeowmeowmeowmeow/gtsrb-german-traffic-sign"

    print(f"Đang tải dataset từ Kaggle: {DATASET}")
    print("(Khoảng 300MB, thường mất 1-2 phút)...")

    api.dataset_download_files(DATASET, path=DOWNLOAD_DIR, unzip=False)

    # Giải nén
    zip_path = os.path.join(DOWNLOAD_DIR, "gtsrb-german-traffic-sign.zip")
    print("Đang giải nén...")

    os.makedirs(EXTRACTED_DIR, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(EXTRACTED_DIR)

    print("Giải nén xong!")

    # Xác nhận kết quả
    if os.path.exists(TRAIN_DIR):
        final_classes = [d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))]
        total_imgs = sum(len(os.listdir(os.path.join(TRAIN_DIR, d))) for d in final_classes)
        print(f"Hoàn tất! {len(final_classes)} class, {total_imgs} ảnh trong {TRAIN_DIR}")
    else:
        # Kaggle dataset có thể có cấu trúc khác, liệt kê để kiểm tra
        print("Nội dung thư mục extracted:")
        for item in os.listdir(EXTRACTED_DIR):
            print(f"  {item}")

Chỉ có 34/43 class. Đang tải lại...
Đang tải dataset từ Kaggle: meowmeowmeowmeowmeow/gtsrb-german-traffic-sign
(Khoảng 300MB, thường mất 1-2 phút)...
Dataset URL: https://www.kaggle.com/datasets/meowmeowmeowmeowmeow/gtsrb-german-traffic-sign
Đang giải nén...
Giải nén xong!
Hoàn tất! 43 class, 39209 ảnh trong c:\Users\pct2k\Downloads\DL\GTSRB-DeepLearning-Classification\data\extracted\Train


In [3]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
from torchvision import datasets
from sklearn.model_selection import train_test_split

# Tự động xác định BASE_DIR
if os.path.exists(os.path.join(os.getcwd(), "notebooks")):
    BASE_DIR = os.getcwd()
elif os.path.basename(os.getcwd()) == "notebooks":
    BASE_DIR = str(Path(os.getcwd()).parent)
else:
    BASE_DIR = os.getcwd()

gtsrb_root = os.path.join(BASE_DIR, 'data', 'extracted')

if __name__ == '__main__':

    train_dir = None
    for root, dirs, files in os.walk(gtsrb_root):
        numeric_subdirs = [d for d in dirs if d.isdigit() or (d.startswith('0') and d.replace('0', '').isdigit())]
        if len(numeric_subdirs) >= 40:
            train_dir = root
            break

    if train_dir is None:
        all_folders = []
        for root, dirs, files in os.walk(gtsrb_root):
            for d in dirs:
                if d.isdigit() or (d.startswith('0') and d.replace('0', '').isdigit()):
                    all_folders.append(os.path.join(root, d))
        
        if all_folders:
            unique_parents = set(str(Path(p).parent) for p in all_folders)
            for parent in unique_parents:
                sub_count = len([d for d in os.listdir(parent) if os.path.isdir(os.path.join(parent, d)) and (d.isdigit() or (d.startswith('0') and d.replace('0', '').isdigit()))])
                if sub_count >= 30:
                    train_dir = parent
                    break

    if train_dir is None:
        raise FileNotFoundError(f"Không tìm thấy thư mục chứa các folder số của GTSRB bên trong {gtsrb_root}. Hãy chạy cell tải dataset ở trên trước.")

    print("Thư mục huấn luyện gốc tìm thấy:", train_dir)

    raw_data = datasets.ImageFolder(train_dir)
    img_paths = [Path(img[0]).as_posix() for img in raw_data.imgs]

    targets = [int(Path(p).parent.name) for p in img_paths]

    train_idx, val_idx = train_test_split(
        np.arange(len(targets)),
        test_size=0.2,
        stratify=targets,
        random_state=42
    )

    SPLITS_DIR = os.path.join(BASE_DIR, 'data', 'splits')
    os.makedirs(SPLITS_DIR, exist_ok=True)

    pd.DataFrame({'path': [img_paths[i] for i in train_idx], 'label': [targets[i] for i in train_idx]}).to_csv(os.path.join(SPLITS_DIR, 'Train.csv'), index=False)
    pd.DataFrame({'path': [img_paths[i] for i in val_idx], 'label': [targets[i] for i in val_idx]}).to_csv(os.path.join(SPLITS_DIR, 'val.csv'), index=False)

    unique_classes = sorted(list(set(targets)))
    class_mapping = {int(cls): f"{cls:05d}" if isinstance(cls, int) else str(cls) for cls in unique_classes}

    with open(os.path.join(SPLITS_DIR, 'class_mapping.json'), 'w') as f:
        json.dump(class_mapping, f, indent=4)

    print(f"Đã cập nhật dữ liệu và lưu vào {SPLITS_DIR}!")
    print(f"Tổng kết dữ liệu: Train = {len(train_idx)} ảnh | Val = {len(val_idx)} ảnh | Tổng số lớp = {len(unique_classes)}")

Thư mục huấn luyện gốc tìm thấy: c:\Users\pct2k\Downloads\DL\GTSRB-DeepLearning-Classification\data\extracted\Train
Đã cập nhật dữ liệu và lưu vào c:\Users\pct2k\Downloads\DL\GTSRB-DeepLearning-Classification\data\splits!
Tổng kết dữ liệu: Train = 31367 ảnh | Val = 7842 ảnh | Tổng số lớp = 43


In [ ]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

BASE_DIR = r'C:\Bài\Deep Learning\GTSRB-DeepLearning-Classification-main'
SPLITS_DIR = os.path.join(BASE_DIR, 'data', 'splits')

class GTSRBSharedDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.df = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['path']
        label = int(self.df.iloc[idx]['label'])
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)
        return image, label

if __name__ == '__main__':

    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomRotation(15),
        transforms.ColorJitter(0.2, 0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    train_set = GTSRBSharedDataset(csv_file=os.path.join(SPLITS_DIR, 'train.csv'), transform=train_transform)
    val_set = GTSRBSharedDataset(csv_file=os.path.join(SPLITS_DIR, 'val.csv'), transform=val_transform)

    train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_set, batch_size=32, shuffle=False, num_workers=2)

    print("DataLoader cho tập Train và Val đã xong")

In [ ]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import numpy as np
import matplotlib.pyplot as plt

BASE_DIR = r'C:\Bài\Deep Learning\GTSRB-DeepLearning-Classification-main'
SPLITS_DIR = os.path.join(BASE_DIR, 'data', 'splits')

class GTSRBSharedDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.df = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['path']
        label = int(self.df.iloc[idx]['label'])
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)
        return image, label

if __name__ == '__main__':

    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomRotation(15),
        transforms.ColorJitter(0.2, 0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    train_set = GTSRBSharedDataset(csv_file=os.path.join(SPLITS_DIR, 'train.csv'), transform=train_transform)
    
    train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=0)

    print("Đang lấy dữ liệu mẫu từ train_loader để hiển thị...")
    
    images, labels = next(iter(train_loader))

    fig, axes = plt.subplots(1, 4, figsize=(12, 4))
    for i, ax in enumerate(axes):
        img = images[i].numpy().transpose((1, 2, 0))
        img = np.clip(img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406], 0, 1)

        ax.imshow(img)
        ax.set_title(f"Class ID: {labels[i].item()}")
        ax.axis('off')

    plt.show()